In [2]:
import pandas as pd
df1 = pd.read_csv("dataset/train/train_source1.tsv", sep="\t")

In [30]:
df2 = pd.read_csv("dataset/train/train_source2.tsv", sep="\t")
df3 = pd.read_csv("dataset/train/train_source3.tsv", sep="\t")
test_df1 = pd.read_csv("dataset/test/test_source1.tsv", sep="\t")
test_df2 = pd.read_csv("dataset/test/test_source2.tsv", sep="\t")
test_df3 = pd.read_csv("dataset/test/test_source3.tsv", sep="\t")

In [4]:
df1.info()
df2.info()
df3.info()

<class 'pandas.DataFrame'>
RangeIndex: 2206821 entries, 0 to 2206820
Data columns (total 4 columns):
 #   Column            Dtype
---  ------            -----
 0   entity_id         str  
 1   business_name     str  
 2   business_address  str  
 3   country           str  
dtypes: str(4)
memory usage: 259.3 MB
<class 'pandas.DataFrame'>
RangeIndex: 5034616 entries, 0 to 5034615
Data columns (total 4 columns):
 #   Column            Dtype
---  ------            -----
 0   entity_id         str  
 1   business_name     str  
 2   business_address  str  
 3   country           str  
dtypes: str(4)
memory usage: 602.3 MB
<class 'pandas.DataFrame'>
RangeIndex: 5285603 entries, 0 to 5285602
Data columns (total 4 columns):
 #   Column            Dtype
---  ------            -----
 0   entity_id         str  
 1   business_name     str  
 2   business_address  str  
 3   country           str  
dtypes: str(4)
memory usage: 622.8 MB


In [5]:
df1.isnull().sum()

entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

In [6]:
df2.isnull().sum()

entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

In [7]:
df3.isnull().sum()

entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64

In [8]:
df1.shape[0]

2206821

In [9]:
df2.shape[0]

5034616

In [10]:
df3.shape[0]

5285603

In [11]:
!pip install anyascii                                                                                                            
                                                                                                                                

In [12]:
import unicodedata                                                                                                               
import re                                                                                                                        
from anyascii import anyascii                                                                                                    
import pandas as pd  

In [13]:
def normalize_language(text: str) -> str:                                                                                        
       """                                                                                                                          
       Standardizes Unicode, transliterates non-Latin scripts (Devanagari, Tamil, etc.)                                             
       and accented characters (French é, è, etc.) into clean lowercase Latin text.                                                 
       """                                                                                                                          
       if not isinstance(text, str) or not text.strip() or text.lower() == "nan":                                                   
           return ""                                                                                                                
                                                                                                                                    
       # 1. Unicode decomposition (replaces curly quotes, dashes, ligatures)                                                        
       text = unicodedata.normalize("NFKD", text)                                                                                   
       text = text.replace("’", "'").replace("`", "'").replace("–", "-").replace("—", "-")                                          
                                                                                                                                    
       # 2. Standardize connective symbols before stripping punctuation                                                             
       text = text.replace("&", " and ").replace("@", " at ").replace("+", " plus ")                                                
                                                                                                                                    
       # 3. Transliterate all scripts & accents to clean Latin/ASCII                                                                
       text = anyascii(text)                                                                                                        
                                                                                                                                    
       # 4. Lowercase and clean excess spaces                                                                                       
       text = text.lower().strip()                                                                                                  
       text = re.sub(r"\s+", " ", text)                                                                                             
                                                                                                                                    
       return text                           

In [14]:
test_cases = [                                                                                                                   
       "Raj Investments LLP",                                                                                                       
       "ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி",               # Tamil                                                                              
       "राज इन्वेस्टमेंट्स प्राइवेट लिमिटेड",           # Hindi                                                                              
       "Café & Crème Résidence",                     # French accents                                                               
       "Dréxkor",                                    # Diacritics                                                                   
       "आदित्य प्रॉपर्टीज एलएलपी"                     # Devanagari LLP                                                                 
   ]                                                                                                                                
                                                                                                                                    
for sample in test_cases:                                                                                                        
       print(f"Original: {sample:<35} -> Normalized: {normalize_language(sample)}") 

Original: Raj Investments LLP                 -> Normalized: raj investments llp
Original: ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி     -> Normalized: raj investments elelpi
Original: राज इन्वेस्टमेंट्स प्राइवेट लिमिटेड -> Normalized: raj investmemts praivet limited
Original: Café & Crème Résidence              -> Normalized: cafe and creme residence
Original: Dréxkor                             -> Normalized: drexkor
Original: आदित्य प्रॉपर्टीज एलएलपी            -> Normalized: adity proprtij elelpi


In [32]:
# 1. Normalize business_name
for data in (df1, df2, df3, test_df1, test_df2, test_df3):
       data["clean_name"] = data["business_name"].fillna("").apply(normalize_language)

# 2. Normalize business_address
for data in (df1, df2, df3, test_df1, test_df2, test_df3):
       data["clean_address"] = data["business_address"].fillna("").apply(normalize_language)

print("Language normalization complete for train and test data!")

Language normalization complete for train and test data!


In [16]:
LEGAL_SUFFIXES_REGEX = re.compile(                                                                                               
       r"\b("                                                                                                                       
       # Indic Transliterated Suffixes (produced by anyascii from Hindi/Tamil/etc.)                                                 
       r"elelpi|praivet\s*(limited|ltd)|pra\s*li|praivrr\s*limirrd|"                                                                
       # US / UK / India (English)                                                                                                  
       r"pvt\s*ltd|private\s*limited|pvt|ltd|limited|"                                                                              
       r"llc|l\s*l\s*c|llp|l\s*l\s*p|inc|incorporated|corp|corporation|"                                                            
       r"co|company|"                                                                                                               
       # France (Test set)                                                                                                          
       r"sarl|s\s*a\s*r\s*l|sas|s\s*a\s*s|sasu|sa|s\s*a|eurl|sci|snc|"                                                              
       r"holding|holdings|group|associates"                                                                                         
       r")\b",                                                                                                                      
       re.IGNORECASE                                                                                                                
   )                                                                                                                                
                                                                                                                                    
   # 2. Frequent Transliterated Business Words to English Equivalents                                                               
PHONETIC_WORD_MAP = {                                                                                                            
       r"\bproprtij\b": "properties",                                                                                               
       r"\binvestmemts?\b": "investments",                                                                                          
       r"\bsolyus[a-z]*\b": "solutions",                                                                                            
       r"\bteknoloj[a-z]*\b": "technologies",                                                                                       
       r"\bejemsij\b": "agencies",                                                                                                  
       r"\bimpeks\b": "impex",                                                                                                      
       r"\bphaumdesn\b": "foundation",                                                                                              
       r"\bresturent\b": "restaurant",                                                                                              
       r"\bknstrksns\b": "constructions",                                                                                           
       r"\bphuds\b": "foods",                                                                                                       
       r"\bprodakts\b": "products",                                                                                                 
       r"\badity\b": "aditya",                                                                                                      
       r"\benterpraij[a-z]*\b": "enterprises",                                                                                      
       r"\bmedikl\b": "medical",                                                                                                    
   }                                                                                                                                
                                                                                                                                    
   # 3. URL Prefixes and Domain Extensions                                                                                          
DOMAIN_REGEX = re.compile(r"\.(com|org|net|in|fr|co|io|biz|info|gov|us)\b", re.IGNORECASE)                                       
URL_PREFIX_REGEX = re.compile(r"https?://|www\.", re.IGNORECASE)                                                                 
                                                                                                                                    
                                                                                                                                    
def clean_business_name(text: str) -> str:                                                                                       
       """                                                                                                                          
       Cleans URLs, strips punctuation, removes legal suffixes (including elelpi/praivet limited),                                  
       and unifies phonetic transliteration words into standard English.                                                            
       """                                                                                                                          
       if not isinstance(text, str) or not text.strip():                                                                            
           return ""                                                                                                                
                                                                                                                                    
       s = text.lower()                                                                                                             
                                                                                                                                    
       # 1. Clean URLs and domain extensions (e.g. 'wilfordhancock.com' -> 'wilfordhancock')                                        
       s = URL_PREFIX_REGEX.sub("", s)                                                                                              
       s = DOMAIN_REGEX.sub(" ", s)                                                                                                 
                                                                                                                                    
       # 2. Remove punctuation and special symbols                                                                                  
       s = re.sub(r"[^\w\s]", " ", s)                                                                                               
                                                                                                                                    
       # 3. Strip legal/corporate suffixes (e.g. 'elelpi', 'llc', 'pvt ltd', 'sarl')                                                
       s = LEGAL_SUFFIXES_REGEX.sub(" ", s)                                                                                         
                                                                                                                                    
       # 4. Standardize common phonetic transliterations (e.g. 'proprtij' -> 'properties')                                          
       for pattern, repl in PHONETIC_WORD_MAP.items():                                                                              
           s = re.sub(pattern, repl, s)                                                                                             
                                                                                                                                    
       # 5. Remove extra whitespace                                                                                                 
       return re.sub(r"\s+", " ", s).strip()    

In [17]:
test_samples = [                                                                                                                 
       # Hindi/Tamil transliteration with 'elelpi' & 'proprtij'                                                                     
       ("Aditya Properties LLP", "adity proprtij elelpi"),                                                                          
                                                                                                                                    
       # Tamil transliteration with 'elelpi'                                                                                        
       ("Raj Investments LLP", "raj investments elelpi"),                                                                           
                                                                                                                                    
       # Hindi transliteration with 'praivet limited' & 'investmemts'                                                               
       ("Shyam Investments Pvt Ltd", "syam investmemts praivet limited"),                                                           
                                                                                                                                    
       # Domain names                                                                                                               
       ("wilfordhancock.com", "wilfordhancock"),                                                                                    
                                                                                                                                    
       # France legal entity SARL                                                                                                   
       ("Boulangerie Saint Honore SARL", "boulangerie saint honore")                                                                
   ]                                                                                                                                
                                                                                                                                    
for s1, s2 in test_samples:                                                                                                      
       c1 = clean_business_name(s1)                                                                                                 
       c2 = clean_business_name(s2)                                                                                                 
       match = (c1 == c2)                                                                                                           
       print(f"S1: {c1:<25} | S2/S3: {c2:<25} | Match: {match}")  

S1: aditya properties         | S2/S3: aditya properties         | Match: True
S1: raj investments           | S2/S3: raj investments           | Match: True
S1: shyam investments         | S2/S3: syam investments          | Match: False
S1: wilfordhancock            | S2/S3: wilfordhancock            | Match: True
S1: boulangerie saint honore  | S2/S3: boulangerie saint honore  | Match: True


In [18]:
                                                                                      
df1["clean_name"] = df1["clean_name"].fillna("").apply(clean_business_name)                                                                                                                             
df2["clean_name"] = df2["clean_name"].fillna("").apply(clean_business_name)                                         
df3["clean_name"] = df3["clean_name"].fillna("").apply(clean_business_name)                                         
                                                                                                                                    
print("Name normalization complete!")          

Name normalization complete!


In [19]:
df1.head(10)

,entity_id,business_name,business_address,country,clean_name,clean_address
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop,"1795 westchester drive, high point, nc"
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money,"17560 ellis road, tahlequah, ok"
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,b plus retail,"1712 montebello avenue, phoenix, az"
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,christ chapel,"2100 cameron drive, unit apartment g, dundalk, md"
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,prabhav business center,"797, lake town block a, kolkata, howrah, west ..."
5,S1-851869949,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",US,custom wealth services,"oh, columbus, 5559 orville avenue"
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India,consulting nyasa nursing,"2505, tower 1, oakwood, runwal greens, mulund ..."
7,S1-27541239,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",US,nexus anchor rain,"1111 church street, unit 2007, nashville, tn"
8,S1-629417405,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",US,moore bitwise,"337 oakland avenue, michigan city, in"
9,S1-22305073,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",US,dermatology green medicine,"294 meadowcreek drive, unit unit 2, village of..."


In [20]:
df1.tail(10)

,entity_id,business_name,business_address,country,clean_name,clean_address
2206811,S1-178574324,KD Emera LLC,"425 Cr 6812, Natalia, TX",US,kd emera,"425 cr 6812, natalia, tx"
2206812,S1-668548746,Elliott Institutions,"3644 Dante Road, VA, Russell County",US,elliott institutions,"3644 dante road, va, russell county"
2206813,S1-395658450,"Penelope H. Papesh, P.A.","18192 Business 13, Unit STE J, Branson West, MO",US,penelope h papesh p a,"18192 business 13, unit ste j, branson west, mo"
2206814,S1-199645528,Glyphent Stock,"22349 255th Street, Maple Valley, WA",US,glyphent stock,"22349 255th street, maple valley, wa"
2206815,S1-419336635,Enoch Beacon Annaly LLC,"6002 Green Falls Drive, Houston, TX",US,enoch beacon annaly,"6002 green falls drive, houston, tx"
2206816,S1-479751630,Arlyn Farooqi Prairie Medical,"16804 Stevenage Street, Surprise, AZ",US,arlyn farooqi prairie medical,"16804 stevenage street, surprise, az"
2206817,S1-392755726,Patriot Fund,"MD, 7709 Ashdale Road, Capitol Heights",US,patriot fund,"md, 7709 ashdale road, capitol heights"
2206818,S1-716147977,Sloan Fact of Yonkers,"28 Armstrong Avenue, Unit Apartment 2, Yonkers...",US,sloan fact of yonkers,"28 armstrong avenue, unit apartment 2, yonkers..."
2206819,S1-770136229,Ritter's Machine Works,"5609 Spring Meadow Road, Austin, TX",US,ritter s machine works,"5609 spring meadow road, austin, tx"
2206820,S1-252639948,Abc Sangh Limited,"C/O. Yuken India Limited, B-80, 2Nd Cross, 1St...",India,abc sangh,"c/o. yuken india limited, b-80, 2nd cross, 1st..."


In [21]:
ADDRESS_ABBREVIATIONS = {                                                                                                        
       # US / UK / India (English)                                                                                                  
       r"\b(rd|rd\.)\b": "road",                                                                                                    
       r"\b(st|st\.)\b": "street",                                                                                                  
       r"\b(ave|ave\.|av|av\.)\b": "avenue",                                                                                        
       r"\b(dr|dr\.)\b": "drive",                                                                                                   
       r"\b(blvd|blvd\.)\b": "boulevard",                                                                                           
       r"\b(ln|ln\.)\b": "lane",                                                                                                    
       r"\b(ct|ct\.)\b": "court",                                                                                                   
       r"\b(hwy|hwy\.)\b": "highway",                                                                                               
       r"\b(pkwy|pkwy\.)\b": "parkway",                                                                                             
       r"\b(apt|apartment|ste|suite|pmb)\b": "unit",                                                                                
                                                                                                                                    
       # France (Test Set)                                                                                                          
       r"\b(r\.|r|rue)\b": "rue",                                                                                                   
       r"\b(bd|bd\.|bvd|boulevard)\b": "boulevard",                                                                                 
       r"\b(all|all\.|allee)\b": "allee",                                                                                           
       r"\b(imp|imp\.|impasse)\b": "impasse",                                                                                       
       r"\b(pl|pl\.|place)\b": "place",                                                                                             
       r"\b(che|ch\.|chemin)\b": "chemin",                                                                                          
       r"\b(rte|route)\b": "route",                                                                                                 
   }                                                                                                                                
                                                                                                                                    
   # 2. Landmark Noise Regex (Strips landmark phrases common in Indian addresses)                                                   
LANDMARK_REGEX = re.compile(                                                                                                     
       r"\b(near|opp|opposite|behind|beside|next to|in front of|adjacent to)\b\s*[^,]{1,35}(?=,|$)",                                
       re.IGNORECASE                                                                                                                
   )                                                                                                                                
                                                                                                                                    
   # 3. Postal Code Regex (5-digit US/France, 6-digit India)                                                                        
POSTAL_CODE_REGEX = re.compile(r"\b([0-9]{5,6})\b")                                                                              
                                                                                                                                    
                                                                                                                                    
def normalize_address_record(addr: str) -> dict:                                                                                 
       """                                                                                                                          
       Cleans address text, standardizes street abbreviations across countries,                                                     
       strips landmark noise, extracts postal/PIN code, and handles missing (NaN) values.                                           
       """                                                                                                                          
       # Handle NaN or missing addresses                                                                                            
       if not isinstance(addr, str) or not addr.strip() or addr.lower() == "nan":                                                   
           return {                                                                                                                 
               "address_clean": "",                                                                                                 
               "postal_code": "",                                                                                                   
               "has_address": False                                                                                                 
           }                                                                                                                        
                                                                                                                                    
       # 1. Extract postal code (if present) before stripping numbers                                                               
       codes = POSTAL_CODE_REGEX.findall(addr)                                                                                      
       postal_code = codes[-1] if codes else ""                                                                                     
                                                                                                                                    
       s = addr.lower()                                                                                                             
                                                                                                                                    
       # 2. Strip landmark clauses (e.g. 'NEAR RUSHABH PETROL PUMP,')                                                               
       s = LANDMARK_REGEX.sub(" ", s)                                                                                               
                                                                                                                                    
       # 3. Standardize street abbreviations (e.g. 'rd' -> 'road', 'r.' -> 'rue')                                                   
       for pattern, repl in ADDRESS_ABBREVIATIONS.items():                                                                          
           s = re.sub(pattern, repl, s)                                                                                             
                                                                                                                                    
       # 4. Strip punctuation and excessive spaces                                                                                  
       s = re.sub(r"[^\w\s]", " ", s)                                                                                               
       address_clean = re.sub(r"\s+", " ", s).strip()                                                                               
                                                                                                                                    
       return {                                                                                                                     
           "address_clean": address_clean,                                                                                          
           "postal_code": postal_code,                                                                                              
           "has_address": True                                                                                                      
       }                              

In [22]:
sample_addresses = [
       # India with landmark noise
       ("SURAT, NEAR RUSHABH PETROL PUMP, RING ROAD, Gujarat", "India"),                                                            

       # US with street abbreviations and unit
       ("1064 Newton Rd, Unit 11, Iowa City, IA 52242", "US"),

       # France with abbreviated 'R.' (Rue)
       ("63 R. DE DIEPPE, LILLE, Hauts-de-France", "France"),

       # France with 'Allée' (pre-cleaned by language step)
       ("NO. 5 ALLEE DES HETRES, Pornic, Loire-Atlantique", "France"),

       # Missing / NaN address                                                                                                      
       (float("nan"), "US")
   ]                                                                                                                                

print(f"{'Original':<50} | {'Clean Address':<40} | {'Postal':<6} | Has Address?")
print("-" * 115)

for addr, country in sample_addresses:
       res = normalize_address_record(str(addr) if pd.notna(addr) else "")
       print(f"{str(addr)[:48]:<50} | {res['address_clean'][:38]:<40} | {res['postal_code']:<6} | {res['has_address']}")            

Original                                           | Clean Address                            | Postal | Has Address?
-------------------------------------------------------------------------------------------------------------------
SURAT, NEAR RUSHABH PETROL PUMP, RING ROAD, Guja   | surat ring road gujarat                  |        | True
1064 Newton Rd, Unit 11, Iowa City, IA 52242       | 1064 newton road unit 11 iowa city ia    | 52242  | True
63 R. DE DIEPPE, LILLE, Hauts-de-France            | 63 rue de dieppe lille hauts de france   |        | True
NO. 5 ALLEE DES HETRES, Pornic, Loire-Atlantique   | no 5 allee des hetres pornic loire atl   |        | True
nan                                                |                                          |        | False


In [25]:
def apply_address_normalization(df: pd.DataFrame, input_col="clean_address") -> pd.DataFrame:
       print(f"Normalizing addresses for {len(df):,} rows...")
       # Use the transliterated address column from Step 1
       parsed = df["clean_address"].fillna("").apply(normalize_address_record)

       df["address_clean"] = [p["address_clean"] for p in parsed]
       df["postal_code"] = [p["postal_code"] for p in parsed]
       df["has_address"] = [p["has_address"] for p in parsed]
       return df

In [24]:
df1.head(10)

,entity_id,business_name,business_address,country,clean_name,clean_address
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop,"1795 westchester drive, high point, nc"
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money,"17560 ellis road, tahlequah, ok"
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,b plus retail,"1712 montebello avenue, phoenix, az"
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,christ chapel,"2100 cameron drive, unit apartment g, dundalk, md"
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,prabhav business center,"797, lake town block a, kolkata, howrah, west ..."
5,S1-851869949,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",US,custom wealth services,"oh, columbus, 5559 orville avenue"
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India,consulting nyasa nursing,"2505, tower 1, oakwood, runwal greens, mulund ..."
7,S1-27541239,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",US,nexus anchor rain,"1111 church street, unit 2007, nashville, tn"
8,S1-629417405,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",US,moore bitwise,"337 oakland avenue, michigan city, in"
9,S1-22305073,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",US,dermatology green medicine,"294 meadowcreek drive, unit unit 2, village of..."


In [33]:
STOPWORDS = {"of", "the", "and", "at", "for", "in", "a", "an", "by", "de", "la", "le", "les", "du", "des"}

US_STATES = {
       "al": "alabama", "ak": "alaska", "az": "arizona", "ar": "arkansas",
       "ca": "california", "co": "colorado", "ct": "connecticut", "de": "delaware",
       "fl": "florida", "ga": "georgia", "hi": "hawaii", "id": "idaho",
       "il": "illinois", "in": "indiana", "ia": "iowa", "ks": "kansas",
       "ky": "kentucky", "la": "louisiana", "me": "maine", "md": "maryland",
       "ma": "massachusetts", "mi": "michigan", "mn": "minnesota", "ms": "mississippi",
       "mo": "missouri", "mt": "montana", "ne": "nebraska", "nv": "nevada",
       "nh": "new hampshire", "nj": "new jersey", "nm": "new mexico", "ny": "new york",
       "nc": "north carolina", "nd": "north dakota", "oh": "ohio", "ok": "oklahoma",
       "or": "oregon", "pa": "pennsylvania", "ri": "rhode island", "sc": "south carolina",
       "sd": "south dakota", "tn": "tennessee", "tx": "texas", "ut": "utah",
       "vt": "vermont", "va": "virginia", "wa": "washington", "wv": "west virginia",
       "wi": "wisconsin", "wy": "wyoming", "dc": "district of columbia"
}

COUNTRY_MAP = {
       "us": "us", "usa": "us", "united states": "us",
       "india": "in", "ind": "in",
       "france": "fr", "fr": "fr"
}

ORDINALS = {
       "1st": "1", "first": "1", "2nd": "2", "second": "2",
       "3rd": "3", "third": "3", "4th": "4", "fourth": "4",
       "5th": "5", "fifth": "5"
}

STATE_REGEX = re.compile(r"\b(" + "|".join(sorted(US_STATES, key=len, reverse=True)) + r")\b")
ORDINAL_REGEX = re.compile(r"\b(" + "|".join(sorted(ORDINALS, key=len, reverse=True)) + r")\b")
UNIT_REGEX = re.compile(r"\bunit\s+\w+\b")


def remove_stopwords(text: str) -> str:
       return " ".join(word for word in str(text).split() if word not in STOPWORDS)


def normalize_address_tokens(text: str) -> str:
       address = str(text).lower()
       address = STATE_REGEX.sub(lambda match: US_STATES[match.group(1)], address)
       address = ORDINAL_REGEX.sub(lambda match: ORDINALS[match.group(1)], address)
       address = UNIT_REGEX.sub(" ", address)
       return re.sub(r"\s+", " ", address).strip()


def process_dataset(df: pd.DataFrame) -> pd.DataFrame:
       if "clean_address" in df.columns:
              df = apply_address_normalization(df)
       elif "address_clean" not in df.columns:
              raise KeyError("Expected clean_address or address_clean column")

       df["address_clean"] = df["address_clean"].apply(normalize_address_tokens)
       df["clean_name"] = df["clean_name"].fillna("").apply(remove_stopwords)
       country_values = df["country"].fillna("").astype(str).str.lower().str.strip()
       df["country_clean"] = country_values.map(COUNTRY_MAP).fillna(country_values)
       df["combined_key"] = (df["clean_name"] + " " + df["address_clean"]).apply(
              lambda text: " ".join(sorted(set(text.split())))
       )
       return df


for data in (df1, df2, df3, test_df1, test_df2, test_df3):
       process_dataset(data)
       data.drop(columns=["postal_code", "clean_address"], inplace=True, errors="ignore")

print("Train and test normalization complete!")

Normalizing addresses for 2,206,821 rows...
Normalizing addresses for 5,034,616 rows...
Normalizing addresses for 5,285,603 rows...
Normalizing addresses for 1,732,544 rows...
Normalizing addresses for 4,887,273 rows...
Normalizing addresses for 5,082,316 rows...
Train and test normalization complete!


In [29]:
df1.head(10)


,entity_id,business_name,business_address,country,clean_name,address_clean,has_address,combined_key
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop,1795 westchester drive high point north carolina,True,1795 barbershop carolina drive high north orel...
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money,17560 ellis road tahlequah oklahoma,True,17560 ellis money oklahoma prime road tahlequah
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,b plus retail,1712 montebello avenue phoenix arizona,True,1712 arizona avenue b montebello phoenix plus ...
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,christ chapel,2100 cameron drive g dundalk maryland,True,2100 cameron chapel christ drive dundalk g mar...
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,prabhav business center,797 lake town block a kolkata howrah west bengal,True,797 a bengal block business center howrah kolk...
5,S1-851869949,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",US,custom wealth services,ohio columbus 5559 orville avenue,True,5559 avenue columbus custom ohio orville servi...
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India,consulting nyasa nursing,2505 tower 1 oakwood runwal greens mulund gore...,True,1 2505 bhandup consulting goreagon greens link...
7,S1-27541239,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",US,nexus anchor rain,1111 church street nashville tennessee,True,1111 anchor church nashville nexus rain street...
8,S1-629417405,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",US,moore bitwise,337 oakland avenue michigan city indiana,True,337 avenue bitwise city indiana michigan moore...
9,S1-22305073,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",US,dermatology green medicine,294 meadowcreek drive 2 village of pewaukee wi...,True,2 294 dermatology drive green meadowcreek medi...
